# Dendritic Rule Cache Demo — Xu 2026 pattern reference

This notebook demonstrates the **two-compartment rule-cache pattern**
inspired by Xu, N., et al. *Science* (2026-10-16), DOI:
`10.1126/science.adz5323`.

**Not project core.** flybrain-connectome is a neuron-level combinatorial
algebra. Xu 2026 lives at the subcellular compartment level and is cited
only as an external theoretical anchor (see `docs/XU2026_DENDRITIC_COMPUTATION_IMPACT_2026-10-18.md`
and the E4 B-14 compartment boundary).

The reference implementation is `tools/dendritic_rule_engine.py`.

## The three concepts to track

| Xu 2026 observation | Code object |
|---|---|
| 顶端树突 = 抽象规则缓存 | `RuleCache` (class) / `Rule` (dataclass) |
| 胞体 = 感觉信息处理 | `SensoryStream` |
| 双区室非线性整合产出决策 | `NonlinearIntegrator` |
| 多次验证后才固化为突触 | `Rule.confidence` + `Rule.is_consolidated` |


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../tools'))
from dendritic_rule_engine import DendriticAgent
agent = DendriticAgent(feature_dim=4, consolidation_threshold=3)
agent

## Step 1 — Author three candidate rules

Each rule is a small callable that maps a feature vector to a label. Xu
2026 shows that a single neuron can *hold* multiple such candidates and let
the dendrite/soma interaction select between them.

In [ ]:
agent.add_rules([
    ("rule_high_temperature", lambda f: 1 if f[0] > 0.5 else 0),
    ("rule_high_humidity",    lambda f: 1 if f[1] > 0.5 else 0),
    ("rule_and",              lambda f: 1 if (f[0] > 0.5 and f[1] > 0.5) else 0),
])
agent.dendrite.rules

## Step 2 — Feed a rule-switching stream

The first 12 samples follow rule `rule_and` (both conditions true → label 1).
At sample 13 the rule switches to `rule_high_temperature` (only temperature
matters → label 1 whenever f[0] > 0.5).

This mirrors Xu 2026's auditory task where the hidden category boundary
shifts without warning.

In [ ]:
import random
random.seed(42)

def sample(label_rule: str, label: int) -> tuple:
    if label_rule == "rule_and":
        if label == 1:
            return (random.uniform(0.6, 1.0), random.uniform(0.6, 1.0))
        else:
            return (random.uniform(-1.0, 0.4), random.uniform(-1.0, 0.4))
    if label_rule == "rule_high_temperature":
        if label == 1:
            return (random.uniform(0.6, 1.0), random.uniform(-1.0, 1.0))
        else:
            return (random.uniform(-1.0, 0.4), random.uniform(-1.0, 1.0))

trajectory = []
for i in range(12):
    label = 1 if i % 2 == 0 else 0
    x, y = sample("rule_and", label)
    r = agent.observe((x, y), label)
    trajectory.append((i+1, "rule_and", label, r["decision"], round(r["confidence_leader"], 3)))

for i in range(12):
    label = 1 if i % 2 == 0 else 0
    x, y = sample("rule_high_temperature", label)
    r = agent.observe((x, y), label)
    trajectory.append((i+13, "rule_high_temperature", label, r["decision"], round(r["confidence_leader"], 3)))

for row in trajectory:
    print(f"#{row[0]:2d} | rule={row[1]:<22s} | gt={row[2]} | dec={row[3]} | conf={row[4]}")

## Step 3 — Inspect what got consolidated

After 24 observations, three rules should be marked `consolidated` because
each has accumulated ≥3 verified hits.

In [ ]:
stats = agent.decision_stats()
for r in stats["rule_summary"]:
    print(f"{r['name']:<25s} support={r['support']:3d} failure={r['failure']:3d} conf={r['confidence']:.3f} consolidated={r['consolidated']}")
print()
print(f"accuracy={stats['accuracy']:.3f} over {stats['total_observations']} observations")

## Step 4 — Abstain path (Xu's silent-silence analogue)

When the dendrite has no supporting rule, the integrator should *abstain*
rather than force a decision. This mirrors how Xu 2026's silence-of-tree
optogenetic test reveals the dendrite is *necessary* — the analog here is
that a *null* dendrite means the soma alone is insufficient.

In [ ]:
from dendritic_rule_engine import NonlinearIntegrator
integrator = NonlinearIntegrator()
print("no rules at all:", integrator.decide([], soma_active=True))
print("rules but soma inactive:", integrator.decide([("rule_and", 1, 0.9)], soma_active=False))
print("one high-confidence rule:", integrator.decide([("rule_and", 1, 0.9)], soma_active=True))

## What this is NOT

- **Not a neuron simulation.** No spiking, no calcium dynamics, no voltage
  gates. Xu 2026's real nonlinearity is dendritic spine-calcium; our
  sigmoid gate captures only the qualitative property that a matching rule
  is required for the soma signal to produce a decision.
- **Not project architecture.** flybrain-connectome remains a neuron-level
  combinatorial algebra. Introducing subcellular compartments into the core
  is explicitly deferred (see `docs/XU2026_DENDRITIC_COMPUTATION_IMPACT_2026-10-18.md`
  §2.3).
- **Not a benchmark.** The demo uses 24 synthetic samples and three toy
  rules. It exists to make Xu 2026's pattern *legible* to readers who
  otherwise only meet it in prose.

## Further reading

- Xu, N., et al. *Science* (2026-10-16). DOI: 10.1126/science.adz5323
- `docs/XU2026_DENDRITIC_COMPUTATION_IMPACT_2026-10-18.md` — full impact memo
- `docs/E4_BOUNDARY_ATLAS.md` B-14 — compartment boundary registration
- `docs/PAPER_P4_CASE_STUDY.md` §6.7 — Xu 2026 as PR-2b methodological neighbour
